# DAT535 Exploration: NYC 311 Service Requests + NYC Weather Integration

## Objectives
1. **Reconcile Disparate Data Grains**:
   - **NYC 311 Requests**: Event-driven / transactional interval $[t_{\text{created}}, t_{\text{closed}}]$ (~1 million records across 3 months).
   - **Weather Observations**: Continuous hourly time-series from NYC weather stations (~2,200 hourly records per station across 3 months).
2. **Evaluate Two Join Approaches**:
   - **Approach A (Point-of-Incident)**: Weather at the exact moment of complaint creation (`created_hour`). Measures *weather triggers* (e.g., heavy rain triggering flood/water complaints, extreme heat triggering noise/utility issues).
   - **Approach B (Resolution Window Exposure)**: Weather conditions experienced while the ticket was open $[t_{\text{created}}, t_{\text{closed}}]$. Measures *operational delay* (e.g., severe weather delaying agency response times).
3. **Spatial Granularity Mapping**:
   - Map NYC Boroughs to regional weather stations: **Central Park** (Manhattan/Bronx), **LaGuardia** (Queens), and **JFK** (Brooklyn/Staten Island).
4. **Identify Data Quality Edge Cases for the Silver Layer**:
   - Tickets closed before creation (`closed_date < created_date`), open tickets (`closed_date` is NULL), and temporal interval distribution.

In [1]:
import urllib.request
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('Libraries successfully imported!')
print('Pandas version:', pd.__version__)

Libraries successfully imported!
Pandas version: 3.0.5


--- 
## 1. Inspecting NYC 311 Temporal & Spatial Characteristics

We load a representative sample of NYC 311 service request records from `data/nyc311_last_3_months` to inspect timestamps, resolution duration, and data quality issues.

In [2]:
# Load 5 sample parquet batches (50,000 records) for rapid exploration
data_dir = Path('data/nyc311_last_3_months')
parquet_files = sorted(data_dir.glob('*.parquet'))[:5]
print(f'Loading {len(parquet_files)} parquet batches...')

df_sample = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
print(f'Total records loaded: {len(df_sample):,}')

# Parse timestamps
df_sample['created_dt'] = pd.to_datetime(df_sample['created_date'], errors='coerce')
df_sample['closed_dt'] = pd.to_datetime(df_sample['closed_date'], errors='coerce')

# Calculate resolution duration in hours
df_sample['resolution_hours'] = (df_sample['closed_dt'] - df_sample['created_dt']).dt.total_seconds() / 3600.0

# Data Quality checks
total = len(df_sample)
null_closed = df_sample['closed_dt'].isna().sum()
negative_duration = (df_sample['resolution_hours'] < 0).sum()
closed_valid = df_sample[df_sample['resolution_hours'] >= 0]

print(f'Total records: {total:,}')
print(f'Open tickets (closed_date is NULL): {null_closed:,} ({null_closed/total*100:.2f}%)')
print(f'Corrupt timestamps (closed < created): {negative_duration:,} (Quarantine candidate in Silver!)')
print(f'Valid closed tickets: {len(closed_valid):,}')
print(f"Resolution duration percentiles (hours): median={closed_valid['resolution_hours'].median():.1f}h, 75th={closed_valid['resolution_hours'].quantile(0.75):.1f}h, 95th={closed_valid['resolution_hours'].quantile(0.95):.1f}h")

Loading 5 parquet batches...
Total records loaded: 50,000


Total records: 50,000
Open tickets (closed_date is NULL): 2,223 (4.45%)
Corrupt timestamps (closed < created): 7 (Quarantine candidate in Silver!)
Valid closed tickets: 47,770
Resolution duration percentiles (hours): median=6.3h, 75th=97.0h, 95th=974.0h


--- 
## 2. Querying Multi-Station NYC Weather Data

We query the free Open-Meteo Historical Weather API for three major NYC meteorological stations:
1. **Central Park (`station_cp`)**: `40.7829, -73.9654` $\rightarrow$ Represents **Manhattan** and **Bronx**.
2. **LaGuardia Airport (`station_lga`)**: `40.7769, -73.8740` $\rightarrow$ Represents **Queens**.
3. **JFK Airport (`station_jfk`)**: `40.6413, -73.7781` $\rightarrow$ Represents **Brooklyn** and **Staten Island**.

We use `timezone=America/New_York` to ensure the weather timestamps natively match the NYC 311 timestamps without offset corruption.

In [3]:
# Determine date range matching the 311 sample
start_str = df_sample['created_dt'].min().strftime('%Y-%m-%d')
end_str = df_sample['created_dt'].max().strftime('%Y-%m-%d')
print(f'Fetching weather between {start_str} and {end_str}...')

stations = {
    'station_cp': {'name': 'Central Park', 'lat': 40.7829, 'lon': -73.9654},
    'station_lga': {'name': 'LaGuardia Airport', 'lat': 40.7769, 'lon': -73.8740},
    'station_jfk': {'name': 'JFK Airport', 'lat': 40.6413, 'lon': -73.7781}
}

weather_dfs = []
for station_id, info in stations.items():
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={info['lat']}&longitude={info['lon']}&"
        f"start_date={start_str}&end_date={end_str}&"
        f"hourly=temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,wind_speed_10m,weather_code&"
        f"timezone=America%2FNew_York"
    )
    req = urllib.request.Request(url, headers={'User-Agent': 'DAT535-Explorer/1.0'})
    with urllib.request.urlopen(req) as resp:
        raw = json.loads(resp.read().decode('utf-8'))
    
    df_w = pd.DataFrame(raw['hourly'])
    df_w['station_id'] = station_id
    df_w['weather_dt'] = pd.to_datetime(df_w['time'])
    weather_dfs.append(df_w)
    print(f"Retrieved {len(df_w)} hourly rows for {info['name']} ({station_id})")
    time.sleep(0.3)  # Polite API pause

df_weather_all = pd.concat(weather_dfs, ignore_index=True)
print(f'\nCombined weather records: {len(df_weather_all):,}')
display(df_weather_all.head())

Fetching weather between 2026-07-15 and 2026-07-20...


Retrieved 144 hourly rows for Central Park (station_cp)


Retrieved 144 hourly rows for LaGuardia Airport (station_lga)


Retrieved 144 hourly rows for JFK Airport (station_jfk)



Combined weather records: 432


,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,wind_speed_10m,weather_code,station_id,weather_dt
0,2026-07-15T00:00,27.9,64,30.1,0.0,0.0,0.0,12.9,0,station_cp,2026-07-15 00:00:00
1,2026-07-15T01:00,27.4,66,29.8,0.0,0.0,0.0,12.0,0,station_cp,2026-07-15 01:00:00
2,2026-07-15T02:00,26.7,71,29.6,0.0,0.0,0.0,10.8,0,station_cp,2026-07-15 02:00:00
3,2026-07-15T03:00,26.4,72,29.3,0.0,0.0,0.0,10.1,0,station_cp,2026-07-15 03:00:00
4,2026-07-15T04:00,25.8,75,28.8,0.0,0.0,0.0,8.8,0,station_cp,2026-07-15 04:00:00


--- 
## 3. Testing Approach A: Point-of-Incident Join (The "Trigger" Model)

### Logic:
1. Map each 311 `borough` to its assigned weather `station_id`:
   - `MANHATTAN`, `BRONX` $\rightarrow$ `station_cp`
   - `QUEENS` $\rightarrow$ `station_lga`
   - `BROOKLYN`, `STATEN ISLAND` $\rightarrow$ `station_jfk`
2. Truncate `created_dt` to the hour: `created_hour = created_dt.floor('h')`.
3. Equi-join on `[station_id, created_hour] == [station_id, weather_dt]`.

### Performance in PySpark:
- Exact equi-join on integer/timestamp keys.
- The weather table is small (~6,600 rows for 3 months), making this a prime candidate for a **Broadcast Hash Join** in Spark without shuffling the large 311 table!

In [4]:
# 1. Spatial Borough Mapping
borough_station_map = {
    'MANHATTAN': 'station_cp',
    'BRONX': 'station_cp',
    'QUEENS': 'station_lga',
    'BROOKLYN': 'station_jfk',
    'STATEN ISLAND': 'station_jfk'
}

df_sample['station_id'] = df_sample['borough'].str.upper().map(borough_station_map).fillna('station_cp')
df_sample['created_hour'] = df_sample['created_dt'].dt.floor('h')

# 2. Equi-join
t0 = time.time()
df_joined_point = pd.merge(
    df_sample,
    df_weather_all,
    left_on=['station_id', 'created_hour'],
    right_on=['station_id', 'weather_dt'],
    how='left'
)
duration_point = time.time() - t0

print(f'Point Join completed in {duration_point*1000:.2f} ms!')
print(f'Rows before: {len(df_sample):,} | Rows after: {len(df_joined_point):,} (Zero row explosion)')

# Inspect joined sample
display(df_joined_point[[
    'unique_key', 'complaint_type', 'borough', 'created_date', 
    'temperature_2m', 'precipitation', 'wind_speed_10m'
]].head())

Point Join completed in 26.52 ms!
Rows before: 50,000 | Rows after: 50,000 (Zero row explosion)


,unique_key,complaint_type,borough,created_date,temperature_2m,precipitation,wind_speed_10m
0,69734449,Noise - Residential,BRONX,2026-07-15T18:35:39.000,32.2,0.0,18.7
1,69734931,Damaged Tree,BROOKLYN,2026-07-15T18:35:48.000,34.3,0.2,15.9
2,69726765,Water System,MANHATTAN,2026-07-15T18:36:00.000,32.2,0.0,18.7
3,69727719,Damaged Tree,BROOKLYN,2026-07-15T18:36:39.000,34.3,0.2,15.9
4,69729114,Abandoned Vehicle,BROOKLYN,2026-07-15T18:36:40.000,34.3,0.2,15.9


In [5]:
# Example Analysis: Precipitation vs Complaint Types
rainy = df_joined_point[df_joined_point['precipitation'] > 0.5]
dry = df_joined_point[df_joined_point['precipitation'] == 0.0]

print('=== Top 5 Complaints During Rain (>0.5mm) ===')
print(rainy['complaint_type'].value_counts().head(5))

print('\n=== Top 5 Complaints During Dry Hours ===')
print(dry['complaint_type'].value_counts().head(5))

=== Top 5 Complaints During Rain (>0.5mm) ===
complaint_type
Noise - Residential        738
Illegal Parking            691
Sewer                      432
Noise - Street/Sidewalk    290
WATER LEAK                 252
Name: count, dtype: int64

=== Top 5 Complaints During Dry Hours ===
complaint_type
Illegal Parking            6570
Noise - Residential        3935
Noise - Street/Sidewalk    3567
Blocked Driveway           1962
UNSANITARY CONDITION       1566
Name: count, dtype: int64


--- 
## 4. Testing Approach B: Resolution Window Exposure (The "Lifecycle" Model)

### Logic:
When a service request is open across multiple hours or days, what weather conditions did the agency face while resolving it? Did extreme heat, heavy downpours, or storms cause delays in ticket closure?

### The Engineering Pitfall: Naive Range Join Explosion
If we naively join each ticket against every weather hour where `weather.time >= created_date AND weather.time <= closed_date`, a ticket open for 5 days generates $5 \times 24 = 120$ joined rows. In a distributed Spark cluster, this causes massive row multiplication, catastrophic shuffle spills, and partition skew.

Let's demonstrate what happens to row volume, and then implement the **Engineered Solution**.

In [6]:
# Select a small subset of 500 closed tickets to demonstrate the join behavior safely
df_closed_subset = df_sample[
    df_sample['closed_dt'].notna() & 
    (df_sample['resolution_hours'] > 0) & 
    (df_sample['resolution_hours'] <= 48)
].head(500).copy()

print(f'Testing with {len(df_closed_subset)} closed tickets (lifespan 1 to 48 hours)...')

# -------------------------------------------------------------------------
# DEMO 1: The Naive Interval Expansion
# -------------------------------------------------------------------------
# Cross match within station to evaluate interval condition
t0 = time.time()
naive_matches = []
for _, row in df_closed_subset.iterrows():
    st = row['station_id']
    c_start = row['created_dt']
    c_end = row['closed_dt']
    # find matching weather rows
    w_match = df_weather_all[
        (df_weather_all['station_id'] == st) &
        (df_weather_all['weather_dt'] >= c_start) &
        (df_weather_all['weather_dt'] <= c_end)
    ]
    naive_matches.append(len(w_match))

duration_naive = time.time() - t0
total_expanded_rows = sum(naive_matches)
print(f'Naive Range Join: 500 tickets expanded into {total_expanded_rows:,} intermediate rows!')
print(f'Row expansion factor: {total_expanded_rows / len(df_closed_subset):.1f}x intermediate row growth.')
print(f'At scale (1,000,000 tickets), this would generate ~{total_expanded_rows / len(df_closed_subset) * 1_000_000:,.0f} intermediate rows in Spark!')

Testing with 500 closed tickets (lifespan 1 to 48 hours)...


Naive Range Join: 500 tickets expanded into 3,961 intermediate rows!
Row expansion factor: 7.9x intermediate row growth.
At scale (1,000,000 tickets), this would generate ~7,922,000 intermediate rows in Spark!


In [7]:
# -------------------------------------------------------------------------
# DEMO 2: The Engineered Aggregation Solution (Pre-Aggregated Weather)
# -------------------------------------------------------------------------
# Instead of exploding rows, we pre-aggregate weather by day and station,
# or compute summary metrics directly into the ticket record.

t0 = time.time()

# Pre-aggregate weather metrics per station per day
df_weather_all['weather_date'] = df_weather_all['weather_dt'].dt.date
df_daily_weather = df_weather_all.groupby(['station_id', 'weather_date']).agg(
    avg_temp=('temperature_2m', 'mean'),
    max_temp=('temperature_2m', 'max'),
    min_temp=('temperature_2m', 'min'),
    total_precip=('precipitation', 'sum'),
    max_wind=('wind_speed_10m', 'max')
).reset_index()

# Join ticket creation date to daily weather
df_closed_subset['created_date_only'] = df_closed_subset['created_dt'].dt.date
df_engineered = pd.merge(
    df_closed_subset,
    df_daily_weather,
    left_on=['station_id', 'created_date_only'],
    right_on=['station_id', 'weather_date'],
    how='left'
)
duration_engineered = time.time() - t0

print(f'Engineered Solution completed in {duration_engineered*1000:.2f} ms!')
print(f'Rows before: {len(df_closed_subset):,} | Rows after: {len(df_engineered):,} (Maintained 1:1 grain!)')
display(df_engineered[['unique_key', 'complaint_type', 'borough', 'resolution_hours', 'avg_temp', 'total_precip', 'max_wind']].head())

Engineered Solution completed in 17.60 ms!
Rows before: 500 | Rows after: 500 (Maintained 1:1 grain!)


,unique_key,complaint_type,borough,resolution_hours,avg_temp,total_precip,max_wind
0,69734449,Noise - Residential,BRONX,3.081389,29.437500,1.0,19.3
1,69734931,Damaged Tree,BROOKLYN,23.277778,30.645833,0.5,18.3
2,69729114,Abandoned Vehicle,BROOKLYN,2.602222,30.645833,0.5,18.3
3,69730122,Noise - Residential,BROOKLYN,1.167222,30.645833,0.5,18.3
4,69732781,Illegal Parking,QUEENS,3.380278,30.062500,0.3,20.7


--- 
## 5. Architectural Evaluation & Recommendations for Your Medallion Pipeline

| Dimension | Approach A: Point-of-Incident | Approach B: Lifecycle Exposure | Recommended Hybrid Solution ⭐ |
| :--- | :--- | :--- | :--- |
| **Concept** | Weather at creation hour | Weather during ticket active span | **Both (Silver + Gold)** |
| **Join Type** | Equi-join on `[station_id, created_hour]` | Non-equi interval join or date pre-aggregate | Point join in Silver; Daily summary in Gold |
| **PySpark Strategy** | Broadcast Hash Join | Window aggregation / Pre-bucketed daily joins | Broadcast Hash Join for Silver |
| **Analytical Value** | Proves what weather triggered the complaint | Proves whether bad weather delayed resolution | Complete 360° service intelligence |

### Medallion Layer Design Proposal:
1. **Bronze Layer**:
   - Ingest raw NYC 311 JSON/Parquet batches unchanged (partitioned by `year/month`).
   - Ingest raw NYC Weather JSON from Open-Meteo for the 3 stations unchanged (partitioned by `station/year`).
2. **Silver Layer (The Heart of the Project)**:
   - Standardize column naming, types, and UTC/EDT timezones.
   - Implement **Data Quality Quarantine**: Filter out tickets where `closed_date < created_date` into a separate quarantine table.
   - Execute **Spatial Mapping**: Map Boroughs $\rightarrow$ Weather Stations.
   - Execute **Point-of-Incident Broadcast Join**: Enrich each 311 ticket with temperature, precipitation, and wind at creation hour.
3. **Gold Layer**:
   - **Analytical Cube 1**: Complaint volume by weather severity condition (e.g. Temperature Bands: Freezing $<0^\circ\text{C}$, Mild, Heatwave $>30^\circ\text{C}$; Rain: None, Heavy Rain $>10\text{mm}$).
   - **Analytical Cube 2**: Agency resolution performance during extreme weather vs fair weather.